# FastAPI Posts Module

This notebook documents the full implementation of the **Posts** feature in a FastAPI application.

It is organized into four sections:

| # | Section | File |
|---|---------|------|
| 1 | ORM Model | `app/models/post.py` |
| 2 | Pydantic Schemas | `app/schemas/post.py` |
| 3 | Service Layer | `app/services/post_service.py` |
| 4 | API Router | `app/routers/posts.py` |

---
## 1. ORM Model — `app/models/post.py`

Defines the SQLAlchemy `Post` model with:
- Core fields: `id`, `title`, `content`, `author_id`, `created_at`, `updated_at`
- Relationships to `User` (author) and `Comment` (top-level + all)
- Cascade delete-orphan on comments

In [ ]:
from sqlalchemy import Column, DateTime, ForeignKey, Integer, String, Text
from sqlalchemy.orm import relationship
from sqlalchemy.sql import func

from app.database import Base


class Post(Base):
    __tablename__ = "posts"

    id = Column(Integer, primary_key=True, index=True)
    title = Column(String(200), nullable=False)
    content = Column(Text, nullable=False)
    author_id = Column(Integer, ForeignKey("users.id"), nullable=False)
    created_at = Column(DateTime(timezone=True), server_default=func.now())
    updated_at = Column(DateTime(timezone=True), onupdate=func.now())

    author = relationship("User", back_populates="posts")

    # Top-level comments only (no parent)
    comments = relationship(
        "Comment",
        back_populates="post",
        cascade="all, delete-orphan",
        primaryjoin="and_(Comment.post_id == Post.id, Comment.parent_id == None)",
        lazy="dynamic",
    )

    # All comments including replies
    all_comments = relationship(
        "Comment",
        cascade="all, delete-orphan",
        foreign_keys="Comment.post_id",
        overlaps="comments",
    )

---
## 2. Pydantic Schemas — `app/schemas/post.py`

Defines the request/response shapes:

| Schema | Purpose |
|--------|---------|
| `PostCreate` | Input for creating a post (`title`, `content`) |
| `PostUpdate` | Partial update — both fields optional |
| `PostResponse` | Full post response including nested `author` |
| `PostListResponse` | Lightweight list item (no `content` or `author` object) |
| `PaginatedPosts` | Paginated wrapper around `PostListResponse` |

In [ ]:
from datetime import datetime
from typing import Optional

from pydantic import BaseModel

from app.schemas.user import UserResponse


class PostCreate(BaseModel):
    title: str
    content: str

    class Config:
        str_min_length = 1


class PostUpdate(BaseModel):
    title: Optional[str] = None
    content: Optional[str] = None


class PostResponse(BaseModel):
    id: int
    title: str
    content: str
    author_id: int
    author: UserResponse
    created_at: datetime
    updated_at: Optional[datetime] = None

    model_config = {"from_attributes": True}


class PostListResponse(BaseModel):
    id: int
    title: str
    author_id: int
    created_at: datetime

    model_config = {"from_attributes": True}


class PaginatedPosts(BaseModel):
    total: int
    page: int
    size: int
    pages: int
    items: list[PostListResponse]

---
## 3. Service Layer — `app/services/post_service.py`

Business logic separated from routing:

| Function | Description |
|----------|-------------|
| `get_all_posts` | Paginated query, ordered by `created_at DESC` |
| `get_post_by_id` | Fetch by ID or raise `404` |
| `create_post` | Insert post, commit, and refresh |
| `update_post` | Partial update with ownership check |
| `delete_post` | Admin-only hard delete |
| `_assert_ownership_or_admin` | Shared authorization helper |

In [ ]:
import math

from fastapi import HTTPException, status
from sqlalchemy.orm import Session

from app.models.post import Post
from app.models.user import User, UserRole
from app.schemas.post import PaginatedPosts, PostCreate, PostListResponse, PostUpdate


def get_all_posts(db: Session, page: int = 1, size: int = 10) -> PaginatedPosts:
    offset = (page - 1) * size
    total = db.query(Post).count()
    posts = db.query(Post).order_by(Post.created_at.desc()).offset(offset).limit(size).all()
    return PaginatedPosts(
        total=total,
        page=page,
        size=size,
        pages=math.ceil(total / size) if total else 0,
        items=[PostListResponse.model_validate(p) for p in posts],
    )


def get_post_by_id(db: Session, post_id: int) -> Post:
    post = db.query(Post).filter(Post.id == post_id).first()
    if not post:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Post not found")
    return post


def create_post(db: Session, data: PostCreate, current_user: User) -> Post:
    post = Post(title=data.title, content=data.content, author_id=current_user.id)
    db.add(post)
    db.commit()
    db.refresh(post)
    return post


def update_post(db: Session, post_id: int, data: PostUpdate, current_user: User) -> Post:
    post = get_post_by_id(db, post_id)
    _assert_ownership_or_admin(post.author_id, current_user)
    if data.title is not None:
        post.title = data.title
    if data.content is not None:
        post.content = data.content
    db.commit()
    db.refresh(post)
    return post


def delete_post(db: Session, post_id: int, current_user: User) -> None:
    post = get_post_by_id(db, post_id)
    if current_user.role != UserRole.ADMIN:
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Only admins can delete posts",
        )
    db.delete(post)
    db.commit()


def _assert_ownership_or_admin(author_id: int, current_user: User) -> None:
    if current_user.role != UserRole.ADMIN and current_user.id != author_id:
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Not authorized to modify this resource",
        )

---
## 4. API Router — `app/routers/posts.py`

FastAPI route definitions with role-based access:

| Method | Path | Auth | Description |
|--------|------|------|-------------|
| `GET` | `/posts/` | Public | List posts (paginated) |
| `GET` | `/posts/{id}` | Public | Get single post |
| `POST` | `/posts/` | Author / Admin | Create post |
| `PUT` | `/posts/{id}` | Author (own) / Admin | Update post |
| `DELETE` | `/posts/{id}` | Admin only | Delete post |

In [ ]:
from fastapi import APIRouter, Depends, Query, status
from sqlalchemy.orm import Session

from app.core.dependencies import get_current_user, require_admin, require_author_or_admin
from app.database import get_db
from app.models.user import User
from app.schemas.post import PaginatedPosts, PostCreate, PostResponse, PostUpdate
from app.services import post_service

router = APIRouter(prefix="/posts", tags=["Posts"])


@router.get("/", response_model=PaginatedPosts)
def list_posts(
    page: int = Query(1, ge=1),
    size: int = Query(10, ge=1, le=100),
    db: Session = Depends(get_db),
):
    """Public: list all posts with pagination."""
    return post_service.get_all_posts(db, page, size)


@router.get("/{post_id}", response_model=PostResponse)
def get_post(post_id: int, db: Session = Depends(get_db)):
    """Public: get a single post by ID."""
    return post_service.get_post_by_id(db, post_id)


@router.post("/", response_model=PostResponse, status_code=status.HTTP_201_CREATED)
def create_post(
    data: PostCreate,
    db: Session = Depends(get_db),
    current_user: User = Depends(require_author_or_admin),
):
    """Author/Admin: create a new post."""
    return post_service.create_post(db, data, current_user)


@router.put("/{post_id}", response_model=PostResponse)
def update_post(
    post_id: int,
    data: PostUpdate,
    db: Session = Depends(get_db),
    current_user: User = Depends(require_author_or_admin),
):
    """Author (own posts) / Admin: update a post."""
    return post_service.update_post(db, post_id, data, current_user)


@router.delete("/{post_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_post(
    post_id: int,
    db: Session = Depends(get_db),
    current_user: User = Depends(require_admin),
):
    """Admin only: delete a post."""
    post_service.delete_post(db, post_id, current_user)